# 00 — Reusable Visium HD pipeline configuration and preflight

This notebook is the **single source of truth** for sample names, folder locations,
image registration, 2-µm QC classes, and bin-level metadata transfer rules.

The example configuration is populated for:

```text
TBIO-8110_VisiumHD-Adagrasib-mouseTumor
```

with 12 samples:

```text
Ada-1, Ada-3R, Ada-4R, Ada-6, Ada-7, Ada-8,
Ada-9, Ada-11R, Ada-12, Ada-14R, Ada-15, Ada-16
```

After editing this notebook for another dataset, run it once. It writes a JSON
configuration and a validated sample manifest that notebooks 01–05 consume.

## Reusable folder contract

The source, QC-Zarr, image, durable-output, temporary-output, and optional custom
function roots are independent:

```python
sourcefiles   # Space Ranger sample folders
zarrfiles     # QC-annotated SpatialData Zarrs
imagefiles    # H&E SVS / TIFF files
outputfiles   # durable derived results
TMPFILES      # scratch, masks, temporary Zarrs, native Proseg Zarrs
functiondirs  # optional custom Python modules
```

## Bin-level `.obs` transfer contract

Notebook 03 supports three explicit variable families:

- **Boolean:** fraction of mapped 2-µm bins that are `True`.
- **Categorical:** dominant category by majority vote; exact ties are labeled
  `mixed`; one fraction column is written per category.
- **Continuous:** one or more user-selected aggregations per variable:
  `sum`, `mean`, or `median`.

The TBIO-8110 example activates `qc_class` as a categorical transfer and leaves
additional boolean/continuous lists empty. Add future variables only in the
`BIN_OBS_TRANSFER_SPEC` block below.


In [1]:
# ---------------------------------------------------------------------
# User configuration — edit this cell for another dataset
# ---------------------------------------------------------------------
from __future__ import annotations

from pathlib import Path
import json

PIPELINE_NAME = "tbio8110_stardist_proseg_resolvi_v1"

sourcefiles = Path(
    "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/individual_samples_analysis"
)
zarrfiles = Path(
    "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/zarr_files"
)
imagefiles = Path(
    "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/images"
)
outputfiles = Path(
    "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/derived_files"
)
tmpfiles = Path(
    "/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8111/tmp"
)
functiondirs = Path(
    "/host_root/nethome/reny28/Projects/Custom_functions/python_functions"
)

SAMPLE_IDS = [
    'C2D15_14_60', 'C2D15_18_68', 'C2D15_22_24', 'C2D15_30_81', 'C2D15_7_93'
    , 'Screen_14_60', 'Screen_18_68', 'Screen_22_24', 'Screen_30_81', 'Screen_7_93'
]

# File/folder names can differ from sample IDs. Edit here when needed.
SAMPLE_SPECS = {
    sample: {
        "zarr_name": f"{sample}.zarr",
        "image_name": f"{sample}.tif",
        "outs_relative": f"{sample}/",
    }
    for sample in SAMPLE_IDS
}

QC_TABLE_KEY = "square_002um"
QC_CLASS_COLUMN = "qc_class"

# Canonical class names used internally.
QC_KEEP_CLASSES = [
    "tissue-high_transcript-high",
    "tissue-high_transcript-low",
]
QC_DROP_CLASSES = [
    "tissue-low_transcript-high",
    "tissue-low_transcript-low",
]

# Preserve the raw value but normalize this known accidental spelling.
QC_CLASS_NORMALIZATION = {
    "trancript": "transcript",
}

# Space Ranger full-resolution pixels -> image level-0 pixels.
# [a, b, c, d, e, f] means:
# image_x = a * pxl_col + b * pxl_row + c
# image_y = d * pxl_col + e * pxl_row + f
# Identity is correct only when Space Ranger used the same unmodified level-0 image.
TENX_TO_IMAGE_AFFINE = {
    sample: [1.0, 0.0, 0.0, 0.0, 1.0, 0.0]
    for sample in SAMPLE_IDS
}

# Use only when the SVS lacks trustworthy OpenSlide MPP metadata.
# Value can be None or [mpp_x, mpp_y]. Do not guess from objective power.
IMAGE_MPP_OVERRIDES = {
    sample: None
    for sample in SAMPLE_IDS
}

# ------------------------------------------------------------------
# Transfer variables from table.obs bins to final Proseg cells
# ------------------------------------------------------------------
BIN_OBS_TRANSFER_SPEC = {
    # "all_bins": all annotated 2-µm bins are eligible for transfer.
    # "qc_keep_only": only retained tissue-high bins are eligible.
    "bin_scope": "all_bins",

    # For each bool variable, notebook 03 writes:
    # <name>_fraction_true and <name>_n_valid_bins.
    "boolean": {
        # Example:
        # "is_tumor": {
        #     "true_values": [True, 1, "true", "True"],
        #     "false_values": [False, 0, "false", "False"],
        # },
    },

    # For each categorical variable, notebook 03 writes:
    # <name>_dominant, <name>_dominant_fraction, <name>_is_tie,
    # <name>_n_valid_bins, and <name>__fraction__<category>.
    "categorical": {
        "qc_class": {
            "tie_label": "mixed",
            # Use an explicit universe so every sample receives the same
            # four qc_class fraction columns even when one class is absent.
            "categories": [
                "tissue-high_transcript-high",
                "tissue-high_transcript-low",
                "tissue-low_transcript-high",
                "tissue-low_transcript-low",
            ],
            "normalize_qc_class": True,
            "max_categories": 20,
        },
        # Example:
        # "pathology_region": {
        #     "tie_label": "mixed",
        #     "categories": None,
        #     "normalize_qc_class": False,
        #     "max_categories": 50,
        # },
    },

    # Values can be a string or a list selected from sum/mean/median.
    "continuous": {
        # "transcript_signal": ["mean", "median"],
        # "bin_umi": "sum",
    },
}

# Barcode/registration safeguards.
MIN_QC_TO_POSITION_OVERLAP = 0.95
MIN_QC_TO_RAW_BARCODE_OVERLAP = 0.95
MIN_QC_KEEP_IMAGE_IN_BOUNDS = 0.95
ALIGNMENT_PREVIEW_MAX_POINTS = 100_000
ALIGNMENT_PREVIEW_MAX_SIDE = 2500
RANDOM_SEED = 17

DERIVED_ROOT = outputfiles / PIPELINE_NAME
TEMP_ROOT = tmpfiles / PIPELINE_NAME
CONFIG_ROOT = DERIVED_ROOT / "00_config"

for directory in [DERIVED_ROOT, TEMP_ROOT, CONFIG_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = CONFIG_ROOT / "pipeline_config.json"
MANIFEST_PATH = CONFIG_ROOT / "sample_manifest.csv"

print("Pipeline config will be written to:", CONFIG_PATH)
print("Temporary root:", TEMP_ROOT)
print("Durable output root:", DERIVED_ROOT)


Pipeline config will be written to: /stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/derived_files/tbio8110_stardist_proseg_resolvi_v1/00_config/pipeline_config.json
Temporary root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8111/tmp/tbio8110_stardist_proseg_resolvi_v1
Durable output root: /stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/derived_files/tbio8110_stardist_proseg_resolvi_v1


## Expected dependencies

Run notebook 00 in the modern spatial environment used for SpatialData and
Proseg post-processing. For `.svs` support, this environment needs OpenSlide.

A durable UV-project installation is typically:

```bash
uv add openslide-python openslide-bin spatialdata scanpy pyarrow
uv sync
```

Notebook 01 runs in the StarDist/TensorFlow environment and also needs
`openslide-python` plus the native OpenSlide library.


In [2]:
# ---------------------------------------------------------------------
# Imports and shared helpers
# ---------------------------------------------------------------------
import gzip
import importlib.metadata
import os
import re
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spatialdata

try:
    import openslide
except Exception as exc:
    raise ImportError(
        "OpenSlide is required for .tif preflight. In this UV project, add "
        "openslide-python and openslide-bin, then restart the kernel."
    ) from exc


CUSTOM_FUNCTION_DIR = Path(functiondirs)
if CUSTOM_FUNCTION_DIR.exists():
    if str(CUSTOM_FUNCTION_DIR) not in sys.path:
        sys.path.insert(0, str(CUSTOM_FUNCTION_DIR))
    print("Custom function directory enabled:", CUSTOM_FUNCTION_DIR)
else:
    warnings.warn(
        f"Configured functiondirs path does not exist: {CUSTOM_FUNCTION_DIR}. "
        "The current notebooks do not require custom modules, so preflight "
        "will continue."
    )

print("Python:", sys.executable)
print("spatialdata:", importlib.metadata.version("spatialdata"))
print("openslide-python:", importlib.metadata.version("openslide-python"))


def normalize_qc_class(values: pd.Series) -> pd.Series:
    result = values.astype("string").str.strip()
    for observed, canonical in QC_CLASS_NORMALIZATION.items():
        result = result.str.replace(observed, canonical, regex=False)
    return result


def apply_affine_xy(x, y, affine):
    a, b, c, d, e, f = [float(v) for v in affine]
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    return a * x + b * y + c, d * x + e * y + f


def read_tissue_positions_and_mpp(outs_path: Path):
    square = outs_path / "binned_outputs" / "square_002um"
    positions_path = square / "spatial" / "tissue_positions.parquet"
    scalefactors_path = square / "spatial" / "scalefactors_json.json"

    if not positions_path.exists():
        raise FileNotFoundError(positions_path)
    if not scalefactors_path.exists():
        raise FileNotFoundError(scalefactors_path)

    positions = pd.read_parquet(positions_path)
    if "barcode" in positions.columns:
        positions = positions.set_index("barcode")
    positions.index = positions.index.astype(str)

    scalefactors = json.loads(scalefactors_path.read_text())
    source_mpp = float(scalefactors["microns_per_pixel"])
    return positions, source_mpp, square


def read_tenx_barcodes(raw_matrix_dir: Path) -> pd.Index:
    for name in ["barcodes.tsv.gz", "barcodes.tsv"]:
        path = raw_matrix_dir / name
        if path.exists():
            opener = gzip.open if path.suffix == ".gz" else open
            with opener(path, "rt") as handle:
                values = [line.rstrip("\n") for line in handle]
            return pd.Index(values, dtype="object")
    raise FileNotFoundError(
        f"No barcodes.tsv(.gz) found under {raw_matrix_dir}"
    )


def get_svs_info(image_path: Path, override):
    detected = openslide.OpenSlide.detect_format(str(image_path))
    if detected is None:
        raise ValueError(f"OpenSlide does not recognize {image_path}")

    with openslide.OpenSlide(str(image_path)) as slide:
        width, height = slide.dimensions
        raw_x = slide.properties.get(openslide.PROPERTY_NAME_MPP_X)
        raw_y = slide.properties.get(openslide.PROPERTY_NAME_MPP_Y)

        if override is not None:
            if len(override) != 2:
                raise ValueError(
                    f"MPP override for {image_path.name} must be [x, y]."
                )
            mpp_x, mpp_y = map(float, override)
            mpp_source = "configured_override"
        else:
            if raw_x is None or raw_y is None:
                raise ValueError(
                    f"{image_path.name} lacks standardized MPP metadata. "
                    "Add a scanner-validated IMAGE_MPP_OVERRIDES entry."
                )
            mpp_x, mpp_y = float(raw_x), float(raw_y)
            mpp_source = "openslide_metadata"

        if not (
            np.isfinite(mpp_x)
            and np.isfinite(mpp_y)
            and mpp_x > 0
            and mpp_y > 0
        ):
            raise ValueError(
                f"Invalid image MPP for {image_path.name}: {(mpp_x, mpp_y)}"
            )

        return {
            "detected_format": detected,
            "width": int(width),
            "height": int(height),
            "mpp_x": float(mpp_x),
            "mpp_y": float(mpp_y),
            "mpp_source": mpp_source,
            "level_count": int(slide.level_count),
            "level_dimensions": [
                [int(w), int(h)] for w, h in slide.level_dimensions
            ],
            "level_downsamples": [
                float(v) for v in slide.level_downsamples
            ],
            "associated_images": list(slide.associated_images.keys()),
        }


def save_alignment_preview(
    image_path: Path,
    aligned: pd.DataFrame,
    output_path: Path,
):
    keep = aligned["qc_keep"].to_numpy(dtype=bool)
    eligible = np.flatnonzero(keep)
    rng = np.random.default_rng(RANDOM_SEED)
    if len(eligible) > ALIGNMENT_PREVIEW_MAX_POINTS:
        eligible = rng.choice(
            eligible,
            ALIGNMENT_PREVIEW_MAX_POINTS,
            replace=False,
        )

    with openslide.OpenSlide(str(image_path)) as slide:
        full_width, full_height = slide.dimensions
        thumbnail = slide.get_thumbnail(
            (ALIGNMENT_PREVIEW_MAX_SIDE, ALIGNMENT_PREVIEW_MAX_SIDE)
        ).convert("RGB")

    array = np.asarray(thumbnail)
    sx = array.shape[1] / full_width
    sy = array.shape[0] / full_height

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(array)
    ax.scatter(
        aligned.iloc[eligible]["image_x_px"] * sx,
        aligned.iloc[eligible]["image_y_px"] * sy,
        s=0.15,
        alpha=0.25,
    )
    ax.set_title(
        f"{image_path.name}: retained tissue-high 2-µm bins on tif level 0"
    )
    ax.set_axis_off()
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def summarize_obs_schema(obs: pd.DataFrame, sample_size=100_000):
    if len(obs) > sample_size:
        sample = obs.sample(sample_size, random_state=RANDOM_SEED)
    else:
        sample = obs

    rows = []
    for column in obs.columns:
        series = sample[column]
        examples = [
            str(v) for v in series.dropna().astype(str).unique()[:5]
        ]
        rows.append(
            {
                "column": str(column),
                "dtype": str(obs[column].dtype),
                "sample_nonnull_fraction": float(series.notna().mean()),
                "sample_n_unique": int(series.nunique(dropna=True)),
                "sample_examples": " | ".join(examples),
            }
        )
    return pd.DataFrame(rows)


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:531: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  left = partial(_left_join_spatialelement_table)
/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:532: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  left_exclusive = partial(_left_

Custom function directory enabled: /host_root/nethome/reny28/Projects/Custom_functions/python_functions
Python: /home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/bin/python
spatialdata: 0.7.2
openslide-python: 1.4.6


In [3]:
#also record environment location
import sys
print(sys.executable)

/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/bin/python


In [4]:
# ---------------------------------------------------------------------
# Validate transfer specification before touching large data
# ---------------------------------------------------------------------
_ALLOWED_CONTINUOUS_MODES = {"sum", "mean", "median"}

if BIN_OBS_TRANSFER_SPEC["bin_scope"] not in {
    "all_bins",
    "qc_keep_only",
}:
    raise ValueError(
        "BIN_OBS_TRANSFER_SPEC['bin_scope'] must be all_bins or qc_keep_only."
    )

for variable, modes in BIN_OBS_TRANSFER_SPEC["continuous"].items():
    modes = [modes] if isinstance(modes, str) else list(modes)
    unexpected = sorted(set(modes) - _ALLOWED_CONTINUOUS_MODES)
    if unexpected:
        raise ValueError(
            f"Unsupported continuous mode(s) for {variable}: {unexpected}"
        )

CONFIG_PAYLOAD = {
    "pipeline_name": PIPELINE_NAME,
    "paths": {
        "sourcefiles": str(sourcefiles),
        "zarrfiles": str(zarrfiles),
        "imagefiles": str(imagefiles),
        "outputfiles": str(outputfiles),
        "tmpfiles": str(tmpfiles),
        "functiondirs": str(functiondirs),
        "derived_root": str(DERIVED_ROOT),
        "temp_root": str(TEMP_ROOT),
        "config_root": str(CONFIG_ROOT),
    },
    "sample_ids": SAMPLE_IDS,
    "sample_specs": SAMPLE_SPECS,
    "qc": {
        "table_key": QC_TABLE_KEY,
        "class_column": QC_CLASS_COLUMN,
        "keep_classes": QC_KEEP_CLASSES,
        "drop_classes": QC_DROP_CLASSES,
        "class_normalization": QC_CLASS_NORMALIZATION,
    },
    "image_registration": {
        "tenx_to_image_affine": TENX_TO_IMAGE_AFFINE,
        "image_mpp_overrides": IMAGE_MPP_OVERRIDES,
    },
    "bin_obs_transfer": BIN_OBS_TRANSFER_SPEC,
    "safeguards": {
        "min_qc_to_position_overlap": MIN_QC_TO_POSITION_OVERLAP,
        "min_qc_to_raw_barcode_overlap": MIN_QC_TO_RAW_BARCODE_OVERLAP,
        "min_qc_keep_image_in_bounds": MIN_QC_KEEP_IMAGE_IN_BOUNDS,
        "random_seed": RANDOM_SEED,
    },
    "pipeline_defaults": {
        "target_mpp": 0.3,
        "stardist_model": "2D_versatile_he",
        "stardist_prob_thresh": 0.02,
        "stardist_nms_thresh": 0.30,
        "minimum_qc_keep_bins_per_nucleus": 1,
        "proseg_voxel_size_um": 2,
        "proseg_diffusion_sigma_near_um": 2,
        "proseg_diffusion_sigma_far_um": 8,
    },
}

CONFIG_PATH.write_text(json.dumps(CONFIG_PAYLOAD, indent=2))
print("Wrote:", CONFIG_PATH)


Wrote: /stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/derived_files/tbio8110_stardist_proseg_resolvi_v1/00_config/pipeline_config.json


In [5]:
# ---------------------------------------------------------------------
# Per-sample preflight
# ---------------------------------------------------------------------
manifest_rows = []
preflight_failures = {}

for sample in SAMPLE_IDS:
    print("\n" + "=" * 90)
    print("Preflight:", sample)
    spec = SAMPLE_SPECS[sample]

    sample_config_root = CONFIG_ROOT / sample
    sample_config_root.mkdir(parents=True, exist_ok=True)

    zarr_path = zarrfiles / spec["zarr_name"]
    image_path = imagefiles / spec["image_name"]
    outs_path = sourcefiles / spec["outs_relative"]
    raw_matrix = (
        outs_path
        / "binned_outputs"
        / "square_002um"
        / "raw_feature_bc_matrix"
    )

    try:
        for path in [zarr_path, image_path, outs_path, raw_matrix]:
            if not path.exists():
                raise FileNotFoundError(path)

        positions, source_mpp, square_2um = (
            read_tissue_positions_and_mpp(outs_path)
        )
        raw_barcodes = read_tenx_barcodes(raw_matrix)
        raw_barcode_set = set(raw_barcodes.astype(str))

        sdata = spatialdata.read_zarr(zarr_path)
        if QC_TABLE_KEY not in sdata.tables:
            raise KeyError(
                f"{QC_TABLE_KEY!r} absent. Tables: {list(sdata.tables)}"
            )
        table = sdata.tables[QC_TABLE_KEY]
        table.obs_names = table.obs_names.astype(str)
        if QC_CLASS_COLUMN not in table.obs:
            raise KeyError(
                f"{QC_CLASS_COLUMN!r} absent from {QC_TABLE_KEY}.obs"
            )

        qc_raw = pd.Series(
            table.obs[QC_CLASS_COLUMN].astype("string").to_numpy(),
            index=table.obs_names,
            name="qc_class_raw",
        )
        qc_canonical = normalize_qc_class(qc_raw)
        qc_canonical.index = qc_raw.index
        qc_canonical.name = "qc_class"

        observed = sorted(qc_canonical.dropna().unique().tolist())
        configured = set(QC_KEEP_CLASSES) | set(QC_DROP_CLASSES)
        unexpected = sorted(set(observed) - configured)
        missing_expected = sorted(configured - set(observed))
        if unexpected:
            raise ValueError(
                f"Unexpected canonical qc_class values: {unexpected}"
            )
        if missing_expected:
            warnings.warn(
                f"{sample}: configured QC classes absent from this sample: "
                f"{missing_expected}. This is allowed but recorded."
            )

        qc_keep = qc_canonical.isin(QC_KEEP_CLASSES)
        if not qc_keep.any():
            raise RuntimeError(
                f"{sample}: no bins match the configured QC_KEEP_CLASSES."
            )

        position_overlap = table.obs_names.isin(positions.index)
        raw_overlap = table.obs_names.isin(raw_barcodes)
        position_fraction = float(position_overlap.mean())
        raw_fraction = float(raw_overlap.mean())

        if position_fraction < MIN_QC_TO_POSITION_OVERLAP:
            raise RuntimeError(
                f"QC-to-position barcode overlap {position_fraction:.3%} is below "
                f"{MIN_QC_TO_POSITION_OVERLAP:.3%}."
            )
        if raw_fraction < MIN_QC_TO_RAW_BARCODE_OVERLAP:
            raise RuntimeError(
                f"QC-to-raw-matrix barcode overlap {raw_fraction:.3%} is below "
                f"{MIN_QC_TO_RAW_BARCODE_OVERLAP:.3%}."
            )

        aligned_positions = positions.reindex(table.obs_names)
        row_px = aligned_positions["pxl_row_in_fullres"].to_numpy(
            dtype=np.float64
        )
        col_px = aligned_positions["pxl_col_in_fullres"].to_numpy(
            dtype=np.float64
        )

        affine = TENX_TO_IMAGE_AFFINE[sample]
        image_x, image_y = apply_affine_xy(col_px, row_px, affine)

        image_info = get_svs_info(
            image_path,
            IMAGE_MPP_OVERRIDES.get(sample),
        )

        finite = np.isfinite(image_x) & np.isfinite(image_y)
        image_in_bounds = (
            finite
            & (image_x >= 0)
            & (image_x < image_info["width"])
            & (image_y >= 0)
            & (image_y < image_info["height"])
        )
        keep_array = qc_keep.to_numpy(dtype=bool)
        keep_in_bounds_fraction = float(
            image_in_bounds[keep_array].mean()
        )
        if keep_in_bounds_fraction < MIN_QC_KEEP_IMAGE_IN_BOUNDS:
            raise RuntimeError(
                f"Only {keep_in_bounds_fraction:.3%} of retained QC bins fall "
                "inside TIF level 0. Inspect the TENX_TO_IMAGE_AFFINE entry."
            )

        aligned = pd.DataFrame(
            {
                "barcode": table.obs_names,
                "qc_class_raw": qc_raw.astype(str).to_numpy(),
                "qc_class": qc_canonical.astype(str).to_numpy(),
                "qc_keep": keep_array,
                "pxl_row_in_fullres": row_px,
                "pxl_col_in_fullres": col_px,
                "proseg_x_um": row_px * source_mpp,
                "proseg_y_um": col_px * source_mpp,
                "image_x_px": image_x,
                "image_y_px": image_y,
                "image_in_bounds": image_in_bounds,
                "present_in_positions": position_overlap,
                "present_in_raw_matrix": raw_overlap,
            }
        )

        aligned_path = sample_config_root / f"{sample}_aligned_qc.parquet"
        aligned.to_parquet(aligned_path, index=False)

        class_counts = (
            aligned.groupby(
                ["qc_class_raw", "qc_class", "qc_keep"],
                dropna=False,
            )
            .size()
            .rename("n_bins")
            .reset_index()
        )
        class_counts.to_csv(
            sample_config_root / f"{sample}_qc_class_counts.csv",
            index=False,
        )

        schema = summarize_obs_schema(table.obs)
        schema.to_csv(
            sample_config_root / f"{sample}_bin_obs_schema.csv",
            index=False,
        )

        preview_path = (
            sample_config_root / f"{sample}_svs_qc_alignment.png"
        )
        save_alignment_preview(image_path, aligned, preview_path)

        report = {
            "sample": sample,
            "zarr_path": str(zarr_path),
            "image_path": str(image_path),
            "outs_path": str(outs_path),
            "raw_matrix": str(raw_matrix),
            "square_2um": str(square_2um),
            "aligned_qc_parquet": str(aligned_path),
            "source_microns_per_pixel": source_mpp,
            "image_info": image_info,
            "tenx_to_image_affine": affine,
            "n_qc_bins": int(table.n_obs),
            "n_qc_keep_bins": int(qc_keep.sum()),
            "n_qc_drop_bins": int((~qc_keep).sum()),
            "qc_to_position_overlap": position_fraction,
            "qc_to_raw_barcode_overlap": raw_fraction,
            "qc_keep_image_in_bounds_fraction": keep_in_bounds_fraction,
            "observed_qc_classes": observed,
            "missing_configured_qc_classes": missing_expected,
            "alignment_preview": str(preview_path),
        }
        report_path = sample_config_root / f"{sample}_preflight.json"
        report_path.write_text(json.dumps(report, indent=2))

        manifest_rows.append(
            {
                "sample": sample,
                "zarr_path": str(zarr_path),
                "image_path": str(image_path),
                "outs_path": str(outs_path),
                "raw_matrix": str(raw_matrix),
                "square_2um": str(square_2um),
                "aligned_qc_parquet": str(aligned_path),
                "preflight_json": str(report_path),
                "source_mpp": source_mpp,
                "image_mpp_x": image_info["mpp_x"],
                "image_mpp_y": image_info["mpp_y"],
                "image_width": image_info["width"],
                "image_height": image_info["height"],
                "qc_bins": int(table.n_obs),
                "qc_keep_bins": int(qc_keep.sum()),
                "qc_drop_bins": int((~qc_keep).sum()),
                "qc_keep_image_in_bounds_fraction": (
                    keep_in_bounds_fraction
                ),
            }
        )

        print(json.dumps(report, indent=2))

    except Exception as exc:
        preflight_failures[sample] = (
            f"{type(exc).__name__}: {exc}"
        )
        print("[FAILED]", sample, preflight_failures[sample])

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(MANIFEST_PATH, index=False)

failure_path = CONFIG_ROOT / "preflight_failures.json"
failure_path.write_text(json.dumps(preflight_failures, indent=2))

print("\nValidated samples:", manifest["sample"].tolist())
print("Failures:", json.dumps(preflight_failures, indent=2))
print("Wrote:", MANIFEST_PATH)

if preflight_failures:
    raise RuntimeError(
        "At least one sample failed preflight. Review the per-sample alignment "
        "preview and failure JSON before running notebook 01."
    )



Preflight: C2D15_14_60


/tmp/ipykernel_2399/4151150167.py:36: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = spatialdata.read_zarr(zarr_path)


{
  "sample": "C2D15_14_60",
  "zarr_path": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/zarr_files/C2D15_14_60.zarr",
  "image_path": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/images/C2D15_14_60.tif",
  "outs_path": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/individual_samples_analysis/C2D15_14_60",
  "raw_matrix": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/individual_samples_analysis/C2D15_14_60/binned_outputs/square_002um/raw_feature_bc_matrix",
  "square_2um": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/individual_samples_analysis/C2D15_14_60/binned_outputs/square_002um",
  "aligned_qc_parquet": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/derived_

/tmp/ipykernel_2399/4151150167.py:36: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = spatialdata.read_zarr(zarr_path)


{
  "sample": "C2D15_18_68",
  "zarr_path": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/zarr_files/C2D15_18_68.zarr",
  "image_path": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/images/C2D15_18_68.tif",
  "outs_path": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/individual_samples_analysis/C2D15_18_68",
  "raw_matrix": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/individual_samples_analysis/C2D15_18_68/binned_outputs/square_002um/raw_feature_bc_matrix",
  "square_2um": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/individual_samples_analysis/C2D15_18_68/binned_outputs/square_002um",
  "aligned_qc_parquet": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/derived_

...

/tmp/ipykernel_2399/4151150167.py:36: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = spatialdata.read_zarr(zarr_path)


{
  "sample": "Screen_30_81",
  "zarr_path": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/zarr_files/Screen_30_81.zarr",
  "image_path": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/images/Screen_30_81.tif",
  "outs_path": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/individual_samples_analysis/Screen_30_81",
  "raw_matrix": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/individual_samples_analysis/Screen_30_81/binned_outputs/square_002um/raw_feature_bc_matrix",
  "square_2um": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/individual_samples_analysis/Screen_30_81/binned_outputs/square_002um",
  "aligned_qc_parquet": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/de

/tmp/ipykernel_2399/4151150167.py:36: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = spatialdata.read_zarr(zarr_path)


{
  "sample": "Screen_7_93",
  "zarr_path": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/zarr_files/Screen_7_93.zarr",
  "image_path": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/images/Screen_7_93.tif",
  "outs_path": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/individual_samples_analysis/Screen_7_93",
  "raw_matrix": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/individual_samples_analysis/Screen_7_93/binned_outputs/square_002um/raw_feature_bc_matrix",
  "square_2um": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/source_files/P-20260522-0001/CA120-1001/Batch1_20260731/individual_samples_analysis/Screen_7_93/binned_outputs/square_002um",
  "aligned_qc_parquet": "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/derived_

RuntimeError: At least one sample failed preflight. Review the per-sample alignment preview and failure JSON before running notebook 01.

## Mandatory visual checkpoint

Open every file matching:

```text
00_config/<sample>/<sample>_svs_qc_alignment.png
```

The retained tissue-high bins must overlay the tissue in the SVS without a
translation, rotation, reflection, or scale error. An in-bounds fraction alone
cannot detect every registration mistake.

For a new dataset, edit `TENX_TO_IMAGE_AFFINE` in the first cell and rerun this
notebook until the overlays are correct.
